# 02 · Confirmation

**This notebook opens the sealed test half.** It runs the specification frozen in
[`FREEZE.md`](../FREEZE.md), amended once (framework-internal arms, §15) and re-rendered.
Nothing here is chosen after the data is seen: every threshold, column set, window, row-set
rule and family member is read from `src/prereg.py` and asserted against the literals in
that document.

**`DRY_RUN` controls whether the seal is broken.** With `DRY_RUN = True` the notebook runs
against the *discovery* rows, which share the schema exactly. That exercises every cell,
every literal's shape, every model fit, and both branches of the eight conditional arms,
without touching the sealed table. The numbers it prints in that mode are **meaningless and
must be discarded** — the discovery half is where the specification was chosen, so a p-value
computed on it tests nothing. The dry run tests the code, not the hypotheses.

Set `DRY_RUN = False` exactly once, and only when the dry run is clean.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2

from src import metrics, prereg, splits
from src.guards import expect_all, expect_n

pd.set_option("display.width", 200)

# ---------------------------------------------------------------------------------------
# THE SWITCH.  True  -> discovery rows, seal untouched, numbers meaningless (dry run).
#              False -> the sealed test half is opened, once, irreversibly.
# ---------------------------------------------------------------------------------------
DRY_RUN = False

CODES = ("2.2", "2.3", "2.5")
CONTROLS = list(metrics.CONTROLS)
TEST_FRAMEWORKS = ("MetaGPT", "Magentic")

def col(code):
    # Label column for a MAST code, as stored in the metric parquet.
    return "code_" + code.replace(".", "_")

print("DRY_RUN =", DRY_RUN)
print("UNSEALED events so far:", len(splits.unseal_events()))
print("log state:", splits.event_counts())

DRY_RUN = False
UNSEALED events so far: 0
log state: {'state': {'SEALED': 9, 'INSPECTED': 2}, 'verification': {'VERIFIED': 1}}


## 1 · Pre-flight: the document is committed and the seal is intact

The frozen specification binds only if it is committed. A dirty `prereg.py` here would mean
the thing being tested is not the thing that was published, so this is a hard stop rather
than a warning.

The dry run asserts **zero** `UNSEALED` events before and after. That assertion is what
makes it a dry run rather than the real thing.

In [2]:
import subprocess

REPO = "../../.."
def git(*args):
    return subprocess.run(["git", "-C", REPO, *args],
                          capture_output=True, text=True, check=True).stdout.strip()

FROZEN_SOURCES = ["Framework_Traces/experiment/src/prereg.py",
                  "Framework_Traces/experiment/src/metrics.py",
                  "Framework_Traces/experiment/src/splits.py",
                  "Framework_Traces/experiment/FREEZE.md"]

dirty = [p for p in FROZEN_SOURCES if git("status", "--porcelain", "--", p)]
assert not dirty, f"refusing to run: frozen sources are uncommitted: {dirty}"
print("frozen sources committed :", len(FROZEN_SOURCES), "checked, 0 dirty")
print("HEAD                     :", git("rev-parse", "--short", "HEAD"))

# FREEZE.md must precede any unseal.  With zero unseal events that is trivially true; the
# assert exists so a re-execution after unsealing cannot silently claim it.
unseal_before = len(splits.unseal_events())
if DRY_RUN:
    assert unseal_before == 0, (
        f"dry run refuses to proceed: {unseal_before} UNSEALED events already exist, so "
        "this notebook is no longer being rehearsed before the seal")
print("UNSEALED before          :", unseal_before)

frozen sources committed : 4 checked, 0 dirty
HEAD                     : 239c094
UNSEALED before          : 0


## 2 · The data

`DRY_RUN = True` reads `data/cache/metrics_discovery.parquet`. `DRY_RUN = False` calls
`splits.load_sealed(unseal=True)`, which appends the first `UNSEALED` line to
`data/sealed/UNSEAL_LOG.txt` — the timestamp the brief asks for — and cannot be undone.

The expected test-half counts are literals from the brief: 2.2 = 142, 2.3 = 129, 2.5 = 222
of 625. They are asserted on the real run and **skipped** in the dry run, where the frame is
the 927-row discovery half and those numbers would legitimately not match. What the dry run
checks instead is the property the real assert depends on: that the label columns exist and
are binary.

In [3]:
EXPECTED_TEST_N = 625
EXPECTED_TEST_POSITIVES = {"2.2": 142, "2.3": 129, "2.5": 222}

if DRY_RUN:
    frame = pd.read_parquet(metrics.METRICS_DISCOVERY_PARQUET)
    splits.assert_discovery_only(frame, "dry run of the confirmation notebook")
    print(f"DRY RUN: {len(frame)} discovery rows -- numbers below are MEANINGLESS")
else:
    frame = splits.load_sealed(unseal=True,
                               note="notebooks/02_confirmation.ipynb, confirmatory run")
    print(f"UNSEALED: {len(frame)} test rows")

print("frameworks:", sorted(frame["framework"].astype(str).unique()))

if not DRY_RUN:
    expect_n(frame, "test-half rows", EXPECTED_TEST_N,
             note="the brief's literal; a different number means the split moved")
    observed_pos = {c: int(frame[col(c)].sum()) for c in CODES}
    expect_all(observed_pos, EXPECTED_TEST_POSITIVES,
               what="test-half positives versus the brief's literals")
    print("positives match the brief:", observed_pos)
else:
    for c in CODES:
        vals = set(frame[col(c)].dropna().unique())
        assert vals <= {0.0, 1.0}, f"{col(c)} is not binary: {vals}"
    print("dry run: label columns present and binary;",
          {c: int(frame[col(c)].sum()) for c in CODES},
          "<- discovery counts, not the brief's")

UNSEALED: 625 test rows
frameworks: ['Magentic', 'MetaGPT']
positives match the brief: {'2.2': 142, '2.3': 129, '2.5': 222}


## 3 · Row sets and the six pooled family members

`ARMS` enumerates the four distinct **row sets**; `FAMILY` the six **members**. The two
within-cell standardised 2.3 members re-use the row sets of members 2 and 4 exactly, because
standardising X within framework cell adds no column and so drops no row — asserted below,
not assumed. Conflating row sets with members would score the same rows twice.

Row-set composition is *reported* here, not asserted against the discovery literals: the
test half holds different frameworks and therefore different n. What is asserted is the
structural invariant, that the standardised members land on the same rows as their
unstandardised twins.

In [4]:
ARMS = [("2.2", "p100"), ("2.3", "p100"),
        ("2.2", prereg.EARLINESS_INFERENTIAL_WINDOW),
        ("2.3", prereg.EARLINESS_INFERENTIAL_WINDOW)]
N_ROW_SETS = 4

FAMILY = [("2.2", "p100", False), ("2.3", "p100", False),
          ("2.2", prereg.EARLINESS_INFERENTIAL_WINDOW, False),
          ("2.3", prereg.EARLINESS_INFERENTIAL_WINDOW, False),
          ("2.3", "p100", True),
          ("2.3", prereg.EARLINESS_INFERENTIAL_WINDOW, True)]
assert len(ARMS) == N_ROW_SETS
assert len(FAMILY) == prereg.BH_FAMILY_SIZE
assert sum(std for _, _, std in FAMILY) == 2 * len(prereg.STANDARDISATION_SENSITIVITY_CODES)

def row_set(code, window, frame):
    mcols = [metrics.metric_column(m, window) for m in prereg.MODEL_COLUMNS[code]]
    return frame.dropna(subset=mcols + CONTROLS + [col(code)])

rows = []
for c, w in ARMS:
    sub = row_set(c, w, frame)
    rows.append({"code": c, "window": w, "n": len(sub),
                 "pos": int(sub[col(c)].sum()),
                 "prev": round(sub[col(c)].mean(), 4),
                 "composition": sub["framework"].astype(str).value_counts().to_dict()})
row_table = pd.DataFrame(rows)
expect_n(row_table, "row sets", N_ROW_SETS,
         note="four row sets, one per (code, window) arm; six members share them")
print(row_table.to_string(index=False))

code window   n  pos   prev                       composition
 2.2   p100 625  142 0.2272 {'MetaGPT': 430, 'Magentic': 195}
 2.3   p100 607  128 0.2109 {'MetaGPT': 428, 'Magentic': 179}
 2.2    p50 625  142 0.2272 {'MetaGPT': 430, 'Magentic': 195}
 2.3    p50 152   60 0.3947                 {'Magentic': 152}


## 4 · The pooled fits

Identical machinery to `01_discovery.ipynb` cell 15 — deliberately, so the confirmation is
the same estimator applied to different rows rather than a reimplementation of it.

`CONFIRMATION_MODE` is `refit`: the specification is frozen, the coefficients are not. What
transfers is the column set, the window, the row-set rule, the test statistic, the control
block and the family; coefficients and intercept are estimated fresh on this half.

In [5]:
print("CONFIRMATION_MODE      :", prereg.CONFIRMATION_MODE)
print("refits coefficients    :", prereg.CONFIRMATION_REFITS_COEFFICIENTS)
print("\nfrozen across halves:")
for m in prereg.CONFIRMATION_FROZEN:
    print("   -", m)
print("\nre-estimated on this half:")
for m in prereg.CONFIRMATION_NOT_FROZEN:
    print("   -", m)

CONFIRMATION_MODE      : refit
refits coefficients    : True

frozen across halves:
   - model column sets per code (MODEL_COLUMNS)
   - window per analysis (localisation p100; earliness EARLINESS_INFERENTIAL_WINDOW)
   - row-set rule per arm (complete cases on that arm's columns at that window)
   - test statistic (likelihood ratio, full model versus control-only model)
   - the control block (log_events, n_agents), identical across all codes
   - family membership, family size, alpha and the BH correction
   - the directional hypothesis per code

re-estimated on this half:
   - regression coefficients, including the control coefficients
   - the intercept and hence the base rate
   - any standardisation constants (see the 2.3 within-cell sensitivity analysis)


In [6]:
def standardise_within_cell(sub, mcols):
    # Within-framework-cell z-scoring of the metric columns only.  Never touches the
    # outcome, so it is label-free: computing test-half cell statistics at unsealing is
    # not a leak.
    out = sub.copy()
    for c in mcols:
        g = out.groupby("framework", observed=True)[c]
        sd = g.transform("std")
        out[c] = (out[c] - g.transform("mean")) / sd.where(sd > 0, np.nan)
    return out

def fit_arm(code, window, frame, standardise=False, rows=None, tag=""):
    # Fit one arm.  `rows` restricts to a boolean mask (the framework-internal arms).
    # Returns a dict; on a fit failure it returns one with `failed` set instead of
    # raising, because a separable within-framework cell must route to the descriptive
    # branch rather than abort the notebook.
    mcols = [metrics.metric_column(m, window) for m in prereg.MODEL_COLUMNS[code]]
    sub = row_set(code, window, frame if rows is None else frame.loc[rows])
    if standardise:
        n_before = len(sub)
        sub = standardise_within_cell(sub, mcols).dropna(subset=mcols)
        assert len(sub) == n_before, (
            f"standardisation dropped {n_before - len(sub)} rows on {code}@{window}{tag}; "
            "members 5/6 no longer share the row sets of 2/4")
    y = sub[col(code)].astype(float)
    base = {"code": code, "window": window, "std": standardise, "tag": tag,
            "n": len(sub), "pos": int(y.sum()), "neg": int(len(y) - y.sum())}
    try:
        ctrl = sm.Logit(y, sm.add_constant(sub[CONTROLS].astype(float),
                                           has_constant="add")).fit(disp=0)
        full = sm.Logit(y, sm.add_constant(sub[mcols + CONTROLS].astype(float),
                                           has_constant="add")).fit(disp=0)
        null = sm.Logit(y, np.ones((len(y), 1))).fit(disp=0)
    except Exception as exc:                      # noqa: BLE001 -- routed, not swallowed
        base.update(failed=f"{type(exc).__name__}: {exc}")
        return base
    lr = 2 * (full.llf - ctrl.llf)
    base.update(
        failed=None,
        converged=bool(full.mle_retvals.get("converged", False)
                       and ctrl.mle_retvals.get("converged", False)),
        lr_stat=lr, lr_df=len(mcols), p_raw=chi2.sf(lr, len(mcols)),
        mcfadden_full=1 - full.llf / null.llf, mcfadden_ctrl=1 - ctrl.llf / null.llf,
        coefs={m: full.params[metrics.metric_column(m, window)]
               for m in prereg.MODEL_COLUMNS[code]},
        pvals={m: full.pvalues[metrics.metric_column(m, window)]
               for m in prereg.MODEL_COLUMNS[code]},
        ses={m: full.bse[metrics.metric_column(m, window)]
             for m in prereg.MODEL_COLUMNS[code]},
        cis={m: tuple(full.conf_int().loc[metrics.metric_column(m, window)])
             for m in prereg.MODEL_COLUMNS[code]})
    return base

fits = [fit_arm(c, w, frame, standardise=s) for c, w, s in FAMILY]
expect_n(fits, "pooled family members fitted", prereg.BH_FAMILY_SIZE,
         note="six members over four row sets; the standardised pair shares the row sets "
              "of members 2 and 4, so no LR test crosses a row set")

failed = [f for f in fits if f["failed"]]
assert not failed, f"pooled members must converge; failures: {failed}"

for a, b in ((1, 4), (3, 5)):
    assert (fits[a]["n"], fits[a]["pos"]) == (fits[b]["n"], fits[b]["pos"]), \
        f"member {b + 1} row set diverged from member {a + 1}"

for i, r in enumerate(fits, start=1):
    tag = "  [within-cell standardised]" if r["std"] else ""
    print(f"member {i}  {r['code']} @{r['window']:4s}  n={r['n']:3d} pos={r['pos']:3d}  "
          f"LR={r['lr_stat']:7.2f} df={r['lr_df']}  p={r['p_raw']:.3e}   McFadden "
          f"full={r['mcfadden_full']:.4f} ctrl={r['mcfadden_ctrl']:.4f} "
          f"(delta {r['mcfadden_full'] - r['mcfadden_ctrl']:+.4f}){tag}")
    for m, b in r["coefs"].items():
        want = prereg.DIRECTIONAL_HYPOTHESES[r["code"]][m]
        got = "positive" if b > 0 else "negative"
        if got != want:
            flag = "   <-- CONTRARY to the declared direction"
        elif m in prereg.RESTATED_DIRECTIONS.get(r["code"], ()):
            flag = "   <-- agrees with a DISCOVERY-INFORMED restatement, not the taxonomy"
        else:
            flag = ""
        print(f"      {m:18s} beta={b:+9.4f} se={r['ses'][m]:7.4f} p={r['pvals'][m]:.4f}  "
              f"declared {want:8s} got {got:8s}{flag}")

member 1  2.2 @p100  n=625 pos=142  LR=   0.53 df=2  p=7.681e-01   McFadden full=0.1494 ctrl=0.1486 (delta +0.0008)
      question_rate      beta=  +0.1449 se= 0.6928 p=0.8343  declared positive got positive   <-- agrees with a DISCOVERY-INFORMED restatement, not the taxonomy
      first_output_pos   beta=  +1.1046 se= 1.5853 p=0.4859  declared negative got positive   <-- CONTRARY to the declared direction
member 2  2.3 @p100  n=607 pos=128  LR=  14.82 df=2  p=6.041e-04   McFadden full=0.1115 ctrl=0.0878 (delta +0.0237)
      min_task_sim       beta=  -5.8312 se= 1.9509 p=0.0028  declared negative got negative
      drift              beta=  +1.9199 se= 0.7359 p=0.0091  declared positive got positive
member 3  2.2 @p50   n=625 pos=142  LR=   3.16 df=2  p=2.061e-01   McFadden full=0.1533 ctrl=0.1486 (delta +0.0047)
      question_rate      beta=  -0.3456 se= 0.7276 p=0.6348  declared positive got negative   <-- CONTRARY to the declared direction
      first_output_pos   beta=  +2.0883 s

## 5 · The eight framework-internal arms, and the gate

Amendment 1 (§15). Each arm is the standardised model on **one test framework's own rows**,
against the same control block. The gate is `prereg.FRAMEWORK_INTERNAL_GATE`: at least 20
positives and 20 negatives; convergence without perfect separation and every metric standard
error below 10 on the standardised scale; non-zero variance in every model column.

The gate reads **row counts and convergence diagnostics only**. The p-value is computed but
not consulted until after `tested` is decided, and the cell below *proves* that ordering — it
re-gates every arm on a copy whose p-value has been destroyed and asserts the verdict is
unchanged — rather than asking the reader to trust it.

In the dry run the frameworks present are ChatDev/AG2 rather than MetaGPT/Magentic, so the
arms are re-pointed at whatever the frame actually holds. That substitution is what lets the
dry run exercise the same code on the same shape.

In [7]:
present = sorted(frame["framework"].astype(str).unique())
if DRY_RUN:
    ARM_FRAMEWORKS = tuple(present[:2])
    print(f"DRY RUN: arms re-pointed from {TEST_FRAMEWORKS} to {ARM_FRAMEWORKS}")
else:
    ARM_FRAMEWORKS = TEST_FRAMEWORKS
    missing = [f for f in ARM_FRAMEWORKS if f not in present]
    assert not missing, f"test half is missing a framework the arms need: {missing}"
    print("arms as declared in prereg.FRAMEWORK_INTERNAL_ARMS:",
          prereg.FRAMEWORK_INTERNAL_ARMS_MAX)

# Arm order follows prereg.FRAMEWORK_INTERNAL_ARMS, with the framework substituted in the
# dry run, so the declared tuple's shape and count are exercised either way.
fw_map = dict(zip(TEST_FRAMEWORKS, ARM_FRAMEWORKS))
INTERNAL_ARMS = [(c, fw_map[f], w) for c, f, w in prereg.FRAMEWORK_INTERNAL_ARMS]
expect_n(INTERNAL_ARMS, "framework-internal arms", prereg.FRAMEWORK_INTERNAL_ARMS_MAX,
         note="2 codes x 2 test frameworks x 2 windows, fixed in Amendment 1")
print("gate literals: min_pos =", prereg.FRAMEWORK_INTERNAL_MIN_POSITIVES,
      " min_neg =", prereg.FRAMEWORK_INTERNAL_MIN_NEGATIVES,
      " max_se =", prereg.FRAMEWORK_INTERNAL_MAX_SE)

arms as declared in prereg.FRAMEWORK_INTERNAL_ARMS: 8
gate literals: min_pos = 20  min_neg = 20  max_se = 10.0


In [8]:
def gate_arm(fit, sub, mcols):
    # Apply prereg.FRAMEWORK_INTERNAL_GATE.  Returns (tested, reasons).  Reads only row
    # counts and convergence diagnostics; fit["p_raw"] is never touched -- asserted below.
    reasons = []
    if fit["pos"] < prereg.FRAMEWORK_INTERNAL_MIN_POSITIVES:
        reasons.append(f"(a) positives {fit['pos']} < "
                       f"{prereg.FRAMEWORK_INTERNAL_MIN_POSITIVES}")
    if fit["neg"] < prereg.FRAMEWORK_INTERNAL_MIN_NEGATIVES:
        reasons.append(f"(a) negatives {fit['neg']} < "
                       f"{prereg.FRAMEWORK_INTERNAL_MIN_NEGATIVES}")
    if fit["failed"]:
        reasons.append(f"(b) fit failed: {fit['failed']}")
    else:
        if not fit["converged"]:
            reasons.append("(b) logit did not converge")
        bad = {m: se for m, se in fit["ses"].items()
               if not np.isfinite(se) or se >= prereg.FRAMEWORK_INTERNAL_MAX_SE}
        if bad:
            reasons.append(f"(b) standard error >= {prereg.FRAMEWORK_INTERNAL_MAX_SE:g}: "
                           + ", ".join(f"{m}={se:.3f}" for m, se in bad.items()))
    flat = [c for c in mcols if sub[c].nunique(dropna=True) < 2]
    if flat:
        reasons.append(f"(c) zero variance: {flat}")
    return (not reasons), reasons

internal = []
for code_, fw, win in INTERNAL_ARMS:
    mask = frame["framework"].astype(str) == fw
    mcols = [metrics.metric_column(m, win) for m in prereg.MODEL_COLUMNS[code_]]
    sub = row_set(code_, win, frame.loc[mask])
    if len(sub) and sub[mcols].notna().any().all():
        fit = fit_arm(code_, win, frame, standardise=True, rows=mask, tag=f" [{fw}]")
    else:
        fit = {"code": code_, "window": win, "std": True, "tag": f" [{fw}]",
               "n": len(sub),
               "pos": int(sub[col(code_)].sum()) if len(sub) else 0,
               "neg": int(len(sub) - sub[col(code_)].sum()) if len(sub) else 0,
               "failed": "no defined rows"}
    fit["framework"] = fw
    fit["tested"], fit["gate_reasons"] = gate_arm(fit, sub, mcols)
    internal.append(fit)

expect_n(internal, "framework-internal arms evaluated",
         prereg.FRAMEWORK_INTERNAL_ARMS_MAX,
         note="every declared arm is evaluated; the gate decides tested vs descriptive")

# Structural proof that the gate did not consult a p-value: re-gate on a copy whose p_raw
# is destroyed and require the identical verdict.
for fit, (code_, fw, win) in zip(internal, INTERNAL_ARMS):
    mask = frame["framework"].astype(str) == fw
    mcols = [metrics.metric_column(m, win) for m in prereg.MODEL_COLUMNS[code_]]
    spoof = dict(fit)
    spoof["p_raw"] = float("nan")
    assert gate_arm(spoof, row_set(code_, win, frame.loc[mask]), mcols)[0] == fit["tested"], \
        "the gate's verdict changed when the p-value was destroyed -- it is reading one"
print("gate verdicts are independent of p (checked by destroying p and re-gating)\n")

for fit in internal:
    head = (f"{fit['code']} @{fit['window']:4s} {fit['framework']:9s} n={fit['n']:3d} "
            f"pos={fit['pos']:3d} neg={fit['neg']:3d}")
    if fit["tested"]:
        print(f"TESTED      {head}  LR={fit['lr_stat']:7.2f} p={fit['p_raw']:.3e}")
    else:
        print(f"DESCRIPTIVE {head}  -> " + "; ".join(fit["gate_reasons"]))
        if not fit["failed"]:
            for m, b in fit["coefs"].items():
                print(f"                 {m:18s} beta={b:+9.4f} "
                      f"se={fit['ses'][m]:7.4f}  (no p-value, no BH slot)")

n_tested = sum(f["tested"] for f in internal)
print(f"\ngate admitted {n_tested} of {prereg.FRAMEWORK_INTERNAL_ARMS_MAX} arms")

gate verdicts are independent of p (checked by destroying p and re-gating)

DESCRIPTIVE 2.2 @p100 MetaGPT   n=430 pos= 65 neg=365  -> (b) logit did not converge
                 question_rate      beta=  +0.1602 se= 0.1214  (no p-value, no BH slot)
                 first_output_pos   beta=  -0.0538 se= 0.1527  (no p-value, no BH slot)
TESTED      2.2 @p100 Magentic  n=195 pos= 77 neg=118  LR=   2.19 p=3.344e-01
DESCRIPTIVE 2.3 @p100 MetaGPT   n=428 pos= 68 neg=360  -> (b) fit failed: LinAlgError: Singular matrix
TESTED      2.3 @p100 Magentic  n=179 pos= 60 neg=119  LR=   0.26 p=8.772e-01
DESCRIPTIVE 2.2 @p50  MetaGPT   n=430 pos= 65 neg=365  -> (b) logit did not converge
                 question_rate      beta=  +0.1214 se= 0.1217  (no p-value, no BH slot)
                 first_output_pos   beta=  +0.0284 se= 0.1353  (no p-value, no BH slot)
TESTED      2.2 @p50  Magentic  n=195 pos= 77 neg=118  LR=   1.44 p=4.857e-01
DESCRIPTIVE 2.3 @p50  MetaGPT   n=  0 pos=  0 neg=  0  -> (a) pos

/Users/tomheinzelmann/Documents/IBM Praxiseinsätze/Pe4 mit Pa 2/Pa 2 Code/MAST-data paper/.venv/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/tomheinzelmann/Documents/IBM Praxiseinsätze/Pe4 mit Pa 2/Pa 2 Code/MAST-data paper/.venv/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/tomheinzelmann/Documents/IBM Praxiseinsätze/Pe4 mit Pa 2/Pa 2 Code/MAST-data paper/.venv/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/tomheinzelmann/Documents/IBM Praxiseinsätze/Pe4 mit Pa 2/Pa 2 Code/M

## 6 · The realised BH family, and the correction

Family size is a conditional integer whose bounds are literals fixed in advance:
`BH_FAMILY_SIZE_MIN` = 6 (no arm admitted) to `BH_FAMILY_SIZE_MAX` = 14 (all eight
admitted). The realised integer is recorded **before** any q-value is read, and asserted
against those bounds.

Descriptive arms carry no p-value into the correction — that is what "no BH slot" means.

In [9]:
family = [dict(f, member=f"pooled {i}") for i, f in enumerate(fits, start=1)]
family += [dict(f, member=f"internal {f['code']}@{f['window']}/{f['framework']}")
           for f in internal if f["tested"]]

realised = len(family)
assert prereg.BH_FAMILY_SIZE_MIN <= realised <= prereg.BH_FAMILY_SIZE_MAX, (
    f"realised family size {realised} is outside the pre-registered bounds "
    f"{prereg.BH_FAMILY_SIZE_MIN}..{prereg.BH_FAMILY_SIZE_MAX}")
assert realised == prereg.BH_FAMILY_SIZE + n_tested
print(f"realised BH family size: {realised}  (= {prereg.BH_FAMILY_SIZE} pooled + "
      f"{n_tested} gated arms; bounds {prereg.BH_FAMILY_SIZE_MIN}.."
      f"{prereg.BH_FAMILY_SIZE_MAX})")

p = np.array([r["p_raw"] for r in family])
expect_n(p, "p values entering BH", realised,
         note="one per family member; descriptive arms contribute none")

order = np.argsort(p)
q = np.empty_like(p)
running = 1.0
for rank, idx in reversed(list(enumerate(order, start=1))):
    running = min(running, p[idx] * len(p) / rank)
    q[idx] = running

for r, qi in zip(family, q):
    r["q"] = qi
    tag = " [std]" if r["std"] else "     "
    print(f"{r['member']:38s} {r['code']} @{r['window']:4s}{tag} p={r['p_raw']:.3e}  "
          f"q={qi:.6e}  {'reject' if qi < prereg.BH_ALPHA else 'retain'} H0")

realised BH family size: 10  (= 6 pooled + 4 gated arms; bounds 6..14)
pooled 1                               2.2 @p100      p=7.681e-01  q=8.534960e-01  retain H0
pooled 2                               2.3 @p100      p=6.041e-04  q=3.020276e-03  reject H0
pooled 3                               2.2 @p50       p=2.061e-01  q=5.644019e-01  retain H0
pooled 4                               2.3 @p50       p=3.951e-01  q=5.644019e-01  retain H0
pooled 5                               2.3 @p100 [std] p=1.438e-04  q=1.438015e-03  reject H0
pooled 6                               2.3 @p50  [std] p=3.951e-01  q=5.644019e-01  retain H0
internal 2.2@p100/Magentic             2.2 @p100 [std] p=3.344e-01  q=5.644019e-01  retain H0
internal 2.3@p100/Magentic             2.3 @p100 [std] p=8.772e-01  q=8.772149e-01  retain H0
internal 2.2@p50/Magentic              2.2 @p50  [std] p=4.857e-01  q=6.071839e-01  retain H0
internal 2.3@p50/Magentic              2.3 @p50  [std] p=3.951e-01  q=5.644019e-01  ret

## 7 · The pre-registered readings

Three conjunctions were fixed in advance and are evaluated mechanically here, so the verdict
is read off the specification rather than composed after the fact:

- **2.3 pooled vs standardised** (`PREDICTION_2_3_TEST_HALF`), including the asymmetry
  caveat: the standardised member's outcome carries no reading unless the unstandardised
  member is significant first.
- **within-framework vs pooled** (`PREDICTION_FRAMEWORK_INTERNAL`), outcomes (a)–(d).
- **the earliness null sentence**, which applies verbatim if the p50 window is not
  significant.

In [10]:
qmap = {(r["code"], r["window"], r["std"]): r["q"]
        for r in family if "framework" not in r}

for w in ("p100", prereg.EARLINESS_INFERENTIAL_WINDOW):
    base_sig = qmap[("2.3", w, False)] < prereg.BH_ALPHA
    std_sig = qmap[("2.3", w, True)] < prereg.BH_ALPHA
    if not base_sig:
        verdict = ("not evaluable: the unstandardised member is not significant, so the "
                   "standardised member's outcome carries no reading (asymmetry caveat). "
                   "If both are n.s. this is outcome (c), a null on the primary endpoint, "
                   "and it is reported as one")
    elif std_sig:
        verdict = "outcome (b): the artifact explanation is WRONG (both significant)"
    else:
        verdict = "outcome (a): framework-offset reading supported (pooled sig, std n.s.)"
    print(f"2.3 @{w}: pooled {'sig' if base_sig else 'n.s.'}, "
          f"standardised {'sig' if std_sig else 'n.s.'}\n   -> {verdict}\n")

print(prereg.PREDICTION_2_3_TEST_HALF)

2.3 @p100: pooled sig, standardised sig
   -> outcome (b): the artifact explanation is WRONG (both significant)

2.3 @p50: pooled n.s., standardised n.s.
   -> not evaluable: the unstandardised member is not significant, so the standardised member's outcome carries no reading (asymmetry caveat). If both are n.s. this is outcome (c), a null on the primary endpoint, and it is reported as one

Predicted for MetaGPT and Magentic: the pooled 2.3 model rejects its control model (members 2 and 4 significant after BH), and the within-framework-cell standardised model does not (members 5 and 6 not significant after BH). Outcomes and what each means, fixed in advance: (a) prediction holds -- the framework-offset explanation is supported on a framework pair sharing nothing with the discovery pair, and the finding is that this class of metric carries between-system level information rather than a within-system association with the label; (b) the standardised model also survives -- the artifact exp

In [11]:
internal_tested = [r for r in family if "framework" in r]
within_sig = [r for r in internal_tested if r["q"] < prereg.BH_ALPHA]
pooled_sig = [r for r in family if "framework" not in r and r["q"] < prereg.BH_ALPHA]

if not internal_tested:
    print("No framework-internal arm passed the gate, so PREDICTION_FRAMEWORK_INTERNAL is\n"
          "NOT EVALUABLE: the arms are reported descriptively and the prediction is\n"
          "recorded as untested rather than as failed.")
else:
    w_any, p_any = bool(within_sig), bool(pooled_sig)
    outcome = {
        (True, False): "(a) within-framework significant, pooled not -- the calibration "
                       "reading is SUPPORTED",
        (True, True): "(b) both significant -- pooling is not destructive on this "
                      "framework pair",
        (False, False): "(c) neither significant -- a NULL on the strongest available "
                        "claim, reported as the primary result rather than buried",
        (False, True): "(d) within-framework null, pooled significant -- CONTRADICTS the "
                       "calibration reading, and is reported as contradicting it",
    }[(w_any, p_any)]
    print(f"within-framework significant: {len(within_sig)}/{len(internal_tested)} tested")
    print(f"pooled significant          : {len(pooled_sig)}/{prereg.BH_FAMILY_SIZE}")
    print(f"\n-> {outcome}")

print("\n" + prereg.PREDICTION_FRAMEWORK_INTERNAL)

within-framework significant: 0/4 tested
pooled significant          : 2/6

-> (d) within-framework null, pooled significant -- CONTRADICTS the calibration reading, and is reported as contradicting it

**Predicted: within-framework arms significant after BH, pooled arms not.**

Concretely -- on each test framework separately, the 2.3 metric block rejects its control model (`min_task_sim` negative, `drift` positive), while the pooled 2.3 members 2/4 either fail or, if they pass, dissolve under the pooled within-cell standardisation of members 5/6. That is the calibration reading: the metrics carry a within-system association with the label, and pooling across systems destroys it because the between-system level difference dominates.

Outcomes fixed in advance:

- **(a) within-framework significant and pooled not** -- the calibration reading is supported, and the experiment's positive claim is that trajectory metrics localise failure within a system while transferring poorly across syste

In [12]:
early = [r for r in family
         if r["window"] == prereg.EARLINESS_INFERENTIAL_WINDOW and "framework" not in r]
early_sig = [r for r in early if r["q"] < prereg.BH_ALPHA]
print(f"earliness window: {prereg.EARLINESS_INFERENTIAL_WINDOW}   "
      f"significant members: {len(early_sig)}/{len(early)}")

if not early_sig:
    print("\nThe pre-registered null sentence applies verbatim:\n")
    print(prereg.EARLINESS_NULL_SENTENCE)
else:
    print("\nAn earliness claim is licensed at the pre-registered window only; the "
          "descriptive windows carry no inferential weight.")
    print(f"  descriptive windows: {prereg.EARLINESS_DESCRIPTIVE_WINDOWS}")
    print("\n" + prereg.EARLINESS_ROW_SET_CAVEAT)

earliness window: p50   significant members: 0/3

The pre-registered null sentence applies verbatim:

The earliness test was pre-registered at a single window (p50) chosen before the data on definedness grounds. A non-significant result at that window means no earliness claim is supported by this experiment. The prefix curve at p25, p75 and p100 is reported as description only and carries no inferential weight: those windows were declared descriptive in advance, they were not corrected for as part of the family, and an apparent effect at any of them is a hypothesis for a future confirmation on data not used here, not a finding of this study. In particular, significance at p75 would not license the claim that failures are detectable before a run ends.


## 8 · Effect sizes and classification metrics

Odds ratios with confidence intervals, per the brief. In the unstandardised members the
metrics are on their raw scales, so the OR is per unit of that metric and the unit is named
rather than left implicit.

Classification metrics are **balanced accuracy, macro F1 and Matthews correlation
coefficient**; plain accuracy is never reported. The cut is rank-based at the arm's own
prevalence, per the brief's preference for rank thresholds over absolute ones. These are
in-sample descriptive fits: they quantify separation on the rows the model was fitted on,
they are not held-out estimates, and they carry no BH slot.

In [13]:
rows = []
for r in family:
    for m, b in r["coefs"].items():
        lo, hi = r["cis"][m]
        rows.append({"member": r["member"], "code": r["code"], "window": r["window"],
                     "std": r["std"], "metric": m,
                     "beta": round(b, 4), "se": round(r["ses"][m], 4),
                     "OR": round(float(np.exp(b)), 4),
                     "OR_lo": round(float(np.exp(lo)), 4),
                     "OR_hi": round(float(np.exp(hi)), 4),
                     "unit": "1 SD within framework cell" if r["std"] else "1 raw unit",
                     "p": round(r["pvals"][m], 6)})
or_table = pd.DataFrame(rows)
expect_n(or_table, "coefficient rows", sum(len(r["coefs"]) for r in family),
         note="one row per (family member x model column)")
print(or_table.to_string(index=False))

                    member code window   std           metric    beta     se     OR  OR_lo    OR_hi                       unit        p
                  pooled 1  2.2   p100 False    question_rate  0.1449 0.6928 1.1559 0.2973   4.4939                 1 raw unit 0.834297
                  pooled 1  2.2   p100 False first_output_pos  1.1046 1.5853 3.0180 0.1350  67.4724                 1 raw unit 0.485946
                  pooled 2  2.3   p100 False     min_task_sim -5.8312 1.9509 0.0029 0.0001   0.1343                 1 raw unit 0.002799
                  pooled 2  2.3   p100 False            drift  1.9199 0.7359 6.8203 1.6121  28.8540                 1 raw unit 0.009084
                  pooled 3  2.2    p50 False    question_rate -0.3456 0.7276 0.7078 0.1701   2.9462                 1 raw unit 0.634838
                  pooled 3  2.2    p50 False first_output_pos  2.0883 1.1614 8.0713 0.8286  78.6232                 1 raw unit 0.072168
                  pooled 4  2.3    p50 False    

In [14]:
from sklearn.metrics import balanced_accuracy_score, f1_score, matthews_corrcoef

def rank_cut(scores, prevalence):
    # Flag the top `prevalence` share by fitted score.  A rank cut needs no scale and no
    # tuned threshold, which is why the brief prefers it.
    k = max(1, int(round(prevalence * len(scores))))
    return (scores >= np.sort(scores)[::-1][k - 1]).astype(int)

rows = []
for r in family:
    mcols = [metrics.metric_column(m, r["window"]) for m in prereg.MODEL_COLUMNS[r["code"]]]
    sub = row_set(r["code"], r["window"],
                  frame if "framework" not in r
                  else frame.loc[frame["framework"].astype(str) == r["framework"]])
    if r["std"]:
        sub = standardise_within_cell(sub, mcols).dropna(subset=mcols)
    y = sub[col(r["code"])].astype(int).to_numpy()
    X = sm.add_constant(sub[mcols + CONTROLS].astype(float), has_constant="add")
    score = np.asarray(sm.Logit(y, X).fit(disp=0).predict(X))
    pred = rank_cut(score, y.mean())
    rows.append({"member": r["member"], "n": len(y), "pos": int(y.sum()),
                 "bal_acc": round(balanced_accuracy_score(y, pred), 4),
                 "macro_f1": round(f1_score(y, pred, average="macro"), 4),
                 "mcc": round(matthews_corrcoef(y, pred), 4)})
score_table = pd.DataFrame(rows)
expect_n(score_table, "scored members", len(family),
         note="one in-sample descriptive score row per family member")
print(score_table.to_string(index=False))
print("\nBalanced accuracy, macro F1 and MCC only -- plain accuracy is not reported.")
print("In-sample and descriptive: not held-out estimates, and no BH slot.")

                    member   n  pos  bal_acc  macro_f1    mcc
                  pooled 1 625  142   0.6856    0.6856 0.3712
                  pooled 2 607  128   0.6683    0.6683 0.3367
                  pooled 3 625  142   0.6811    0.6811 0.3621
                  pooled 4 152   60   0.6833    0.6833 0.3667
                  pooled 5 607  128   0.6683    0.6683 0.3367
                  pooled 6 152   60   0.6833    0.6833 0.3667
internal 2.2@p100/Magentic 195   77   0.7639    0.7639 0.5278
internal 2.3@p100/Magentic 179   60   0.6992    0.6992 0.3983
 internal 2.2@p50/Magentic 195   77   0.7639    0.7639 0.5278
 internal 2.3@p50/Magentic 152   60   0.6833    0.6833 0.3667

Balanced accuracy, macro F1 and MCC only -- plain accuracy is not reported.
In-sample and descriptive: not held-out estimates, and no BH slot.


## 9 · Exploratory: 2.5

2.5 sits **outside the confirmatory family**. It is reported as an instrumentation finding:
on MetaGPT the receiver is not missing from the log, it is logged as absent, so three of the
four 2.5 metrics are undefined in principle rather than merely weak. The one surviving
column, `passive_rate`, has a weak construct fit that was pre-registered as a limitation.
Nothing here enters BH, and a null on it is uninformative.

In [15]:
assert "2.5" not in prereg.CONFIRMATORY_CODES and "2.5" in prereg.EXPLORATORY_CODES
print("2.5 in the confirmatory family:", "2.5" in prereg.CONFIRMATORY_CODES)
print("model columns for 2.5        :", prereg.MODEL_COLUMNS["2.5"])
print()
for w in ("p100", prereg.EARLINESS_INFERENTIAL_WINDOW):
    r = fit_arm("2.5", w, frame)
    if r["failed"]:
        print(f"2.5 @{w}: fit failed ({r['failed']}) -- exploratory, no BH slot")
        continue
    m = prereg.MODEL_COLUMNS["2.5"][0]
    print(f"2.5 @{w}: n={r['n']} pos={r['pos']}  LR={r['lr_stat']:.2f} "
          f"p_uncorrected={r['p_raw']:.4f} (NO BH slot)  {m} "
          f"beta={r['coefs'][m]:+.4f} se={r['ses'][m]:.4f}")

print("\n" + prereg.INSTRUMENTATION_2_5)
print("\n" + prereg.PASSIVE_RATE_LIMITATION)
print("\n" + prereg.RECEIVER_PROVENANCE_LIMITATION)

2.5 in the confirmatory family: False
model columns for 2.5        : ('passive_rate',)

2.5 @p100: n=625 pos=222  LR=3.48 p_uncorrected=0.0621 (NO BH slot)  passive_rate beta=-2.0983 se=1.1398
2.5 @p50: n=623 pos=221  LR=1.10 p_uncorrected=0.2941 (NO BH slot)  passive_rate beta=-0.9281 se=0.8997

MetaGPT is not failing to log an addressee. It is logging, explicitly, that there is not one: `to` is present on all 430 kept events and its only value is the broadcast sentinel {'<all>'}. The receiver is therefore unrecoverable by design of the framework rather than by omission of its logger, and no better parser would recover it. Three of the four 2.5 metrics depend on resolving who a message was addressed to, so on this framework they are not weak -- they are undefined in principle. That is an instrumentation finding about what multi-agent frameworks record, not a transfer failure of the metrics, and it is reported as the former.

The entire 2.5 branch rests on one column, `passive_rate`, a

## 10 · Contrary signs, and hypotheses not supported

`CONTRARY_SIGN_DECISION_RULE` requires that the sentence stating a direction opposes the
pre-registration appears **before** its magnitude. The cell below prints in that order by
construction, so the ordering is a property of the code rather than of the writing.

In [16]:
contrary = []
for r in family:
    for m, b in r["coefs"].items():
        want = prereg.DIRECTIONAL_HYPOTHESES[r["code"]][m]
        got = "positive" if b > 0 else "negative"
        if got != want:
            contrary.append((r, m, b, want, got))

if not contrary:
    print("No coefficient opposes its pre-registered direction.")
else:
    print(f"{len(contrary)} coefficient(s) oppose their pre-registered direction.\n")
    for r, m, b, want, got in contrary:
        print(f"{r['member']}  {r['code']} {m}")
        print(f"   The fitted direction ({got}) OPPOSES the pre-registered direction "
              f"({want}). This does not support the hypothesis, and is not reported as "
              f"confirmation of the MAST code's mechanism.")
        print(f"   Magnitude: beta={b:+.4f} se={r['ses'][m]:.4f} "
              f"p={r['pvals'][m]:.4f} (per-coefficient Wald, uncorrected)")
        print(f"   The member's block LR test carries q={r['q']:.3e}, which is a test of "
              f"the block and NOT of this coefficient; it is quoted here only to locate "
              f"the coefficient, never to license it.")
        if m in prereg.RESTATED_DIRECTIONS.get(r["code"], ()):
            print("   NOTE: this direction was RESTATED from the discovery half, so a "
                  "result agreeing with it is discovery-informed and weaker than a "
                  "taxonomy-derived direction.")
        print()

print(prereg.CONTRARY_SIGN_DECISION_RULE)

7 coefficient(s) oppose their pre-registered direction.

pooled 1  2.2 first_output_pos
   The fitted direction (positive) OPPOSES the pre-registered direction (negative). This does not support the hypothesis, and is not reported as confirmation of the MAST code's mechanism.
   Magnitude: beta=+1.1046 se=1.5853 p=0.4859 (per-coefficient Wald, uncorrected)
   The member's block LR test carries q=8.535e-01, which is a test of the block and NOT of this coefficient; it is quoted here only to locate the coefficient, never to license it.

pooled 3  2.2 question_rate
   The fitted direction (negative) OPPOSES the pre-registered direction (positive). This does not support the hypothesis, and is not reported as confirmation of the MAST code's mechanism.
   Magnitude: beta=-0.3456 se=0.7276 p=0.6348 (per-coefficient Wald, uncorrected)
   The member's block LR test carries q=5.644e-01, which is a test of the block and NOT of this coefficient; it is quoted here only to locate the coefficient, neve

## 11 · Close: seal state

The dry run asserts the seal is still shut. The real run asserts that exactly one `UNSEALED`
event exists — the one this notebook created.

In [17]:
after = len(splits.unseal_events())
print("UNSEALED events:", after)
print("log state      :", splits.event_counts())

if DRY_RUN:
    assert after == unseal_before == 0, (
        f"DRY RUN BROKE THE SEAL: {after} UNSEALED events exist. This is the exact failure "
        "the dry run was written to make impossible.")
    print("\nDRY RUN COMPLETE. The seal is intact and every number above is meaningless.")
    print("Discard them. Set DRY_RUN = False to run the confirmation for real.")
else:
    assert after == 1, (
        f"expected exactly one UNSEALED event, found {after}; a second unseal means this "
        "notebook ran twice, and the document is materially weaker for it")
    print("\nCONFIRMATION COMPLETE. One unseal event, as pre-registered.")

UNSEALED events: 1
log state      : {'state': {'SEALED': 9, 'INSPECTED': 2, 'UNSEALED': 1}, 'verification': {'VERIFIED': 1}}

CONFIRMATION COMPLETE. One unseal event, as pre-registered.
